# Feature Extraction — 64-Cell Grid Analysis
## Programming Decay | Skill Module: Relational Cartographies
**Author:** Jiajia Jiang | UCL Bartlett B-Pro MArch Urban Design RC18

### Purpose
This script extracts six environmental features from a fishnet grid (8×8 = 64 cells)
overlaid on the London Borough of Barking & Dagenham in ArcGIS Pro.

Each cell covers approximately **690m × 525m**. The six features capture the full
spectrum of concrete's environmental burden:

| Feature | Description | Data Source |
|---------|-------------|-------------|
| F1 | Brownfield site count | LBBD Brownfield Register 2024 |
| F2 | Building footprint coverage (%) | OS OpenMap Local |
| F3 | Waste infrastructure count (permitted + historic landfill) | Environment Agency |
| F4 | Flood risk zone coverage (%) | Environment Agency Flood Alert Areas |
| F5 | Population (usual residents) | ONS Census 2021 |
| F6 | Ecological sensitivity area coverage (%) | Natural England |

### Output
- `Grid_Features.csv` — 64 rows × 7 columns (Cell_ID + 6 features)

### Prerequisites
- ArcGIS Pro 3.6 with ArcPy
- Spatial joins (F1–F6) must be pre-computed as feature layers in the active map

In [7]:
import arcpy, csv

out_csv = r"Grid_Features.csv"

data = {}

# F1: Brownfield site count per cell
with arcpy.da.SearchCursor("F1_result", ["Cell_ID", "F1_Brownfield"]) as cur:
    for row in cur:
        data[row[0]] = {"F1": row[1] if row[1] else 0}

# F2: Building footprint coverage (%)
with arcpy.da.SearchCursor("F2_result", ["Cell_ID", "sum_Area_SQUAREMETERS", "Shape_Area"]) as cur:
    for row in cur:
        cid, val, sa = row[0], (row[1] if row[1] else 0), row[2]
        if cid not in data: data[cid] = {}
        data[cid]["F2"] = val / sa * 100 if sa else 0

# F3a: Permitted Waste Facilities
with arcpy.da.SearchCursor("F3a_result", ["Cell_ID", "Polygon_Count"]) as cur:
    for row in cur:
        if row[0] not in data: data[row[0]] = {}
        data[row[0]]["F3a"] = row[1] if row[1] else 0

# F3b: Historic Landfill Sites
with arcpy.da.SearchCursor("F3b_result", ["Cell_ID", "Polygon_Count"]) as cur:
    for row in cur:
        if row[0] not in data: data[row[0]] = {}
        data[row[0]]["F3b"] = row[1] if row[1] else 0

# F4: Flood risk zone coverage (%)
with arcpy.da.SearchCursor("F4_result", ["Cell_ID", "sum_Area_SQUAREMETERS", "Shape_Area"]) as cur:
    for row in cur:
        cid, val, sa = row[0], (row[1] if row[1] else 0), row[2]
        if cid not in data: data[cid] = {}
        data[cid]["F4"] = val / sa * 100 if sa else 0

# F5: Population (usual residents, ONS Census 2021)
with arcpy.da.SearchCursor("F5_result", ["Cell_ID", "sum_USUALRES"]) as cur:
    for row in cur:
        if row[0] not in data: data[row[0]] = {}
        data[row[0]]["F5"] = row[1] if row[1] else 0

# F6: Ecological sensitivity area coverage (%)
with arcpy.da.SearchCursor("F6_result", ["Cell_ID", "sum_Area_SQUAREMETERS", "Shape_Area"]) as cur:
    for row in cur:
        cid, val, sa = row[0], (row[1] if row[1] else 0), row[2]
        if cid not in data: data[cid] = {}
        data[cid]["F6"] = val / sa * 100 if sa else 0

# Write CSV
with open(out_csv, "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["Cell_ID","F1_Brownfield","F2_Building","F3_Waste","F4_Flood","F5_Pop","F6_Sensitive"])
    for cid in sorted(data.keys()):
        d = data[cid]
        w.writerow([
            cid,
            d.get("F1", 0),
            round(d.get("F2", 0), 2),
            d.get("F3a", 0) + d.get("F3b", 0),
            round(d.get("F4", 0), 2),
            d.get("F5", 0),
            round(d.get("F6", 0), 2)
        ])

print("Done! Exported to " + out_csv)

Done! Exported to Grid_Features.csv
